# 06 Cold Start Evaluation

This notebook keeps cold-start populations separate from the main benchmark.

- **Known-item limited context:** the base model receives only 7, 14, 28 or 56 recent days, but it has learned those items during training. This is a context sensitivity check, not cold start.
- **Unseen-item limited-history transfer:** item IDs and all their targets are excluded from model training. At prediction time, only the stated last 7, 14, 28 or 56 observations and their past-only scale are supplied.
- **Unseen item:** all target history for selected item IDs is excluded from training.
- **New combination:** a store-item series is excluded while the same item remains visible in another store.
- **Zero shot:** an unseen item receives no demand history; only shared metadata and known future covariates are available.

## 1. Locate the project

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root located; Phase 2 modules are importable.")

Project root located; Phase 2 modules are importable.


## 2. Run the cold-start suite

Unknown item categories map safely to index zero. The unseen-item assertion verifies that held-out item sales never enter model training. A separate model for each history length cannot learn with the current 56-day context plus 7-day target requirement, so the notebook uses a defensible transfer design trained only on other products. These results are not directly combined with the 100-series headline result because the cohorts differ.

In [2]:
from src.phase2.cold_start_experiments import run_cold_start_suite

cold_metrics, cold_predictions, split = run_cold_start_suite(PROJECT_ROOT)
display(cold_metrics)

,MAE,RMSE,WAPE_percent,pinball_q10,pinball_q50,pinball_q90,mean_pinball_loss,interval_coverage_80_percent,empirical_q10_below_percent,empirical_q90_at_or_below_percent,empirical_q90_above_percent,mean_interval_width,forecast_points,cohort,history_days,number_of_series
0,0.939464,2.227286,61.589792,0.132375,0.469732,0.342511,0.314873,92.857143,0.285714,93.142857,6.857143,3.197429,2800,known_item_limited_context,7,100
1,0.923929,2.146509,60.571295,0.126757,0.461964,0.341393,0.310038,93.785714,0.357143,94.142857,5.857143,3.295071,2800,known_item_limited_context,14,100
2,0.921071,2.125063,60.383985,0.126964,0.460536,0.345832,0.311111,94.000000,0.464286,94.464286,5.535714,3.363679,2800,known_item_limited_context,28,100
3,0.922679,2.114132,60.489347,0.128800,0.461339,0.344168,0.311436,93.928571,0.642857,94.571429,5.428571,3.378250,2800,known_item_limited_context,56,100
4,0.660714,1.206382,80.434783,0.082143,0.330357,0.207024,0.206508,96.428571,0.000000,96.428571,3.571429,2.308333,84,new_store_item_combination,full,3
5,1.167293,2.791131,49.364070,0.225714,0.583647,0.668496,0.492619,90.225564,0.000000,90.225564,9.774436,7.116917,532,unseen_item_limited_history_transfer,7,19
6,1.111842,2.570893,47.019078,0.225883,0.555921,0.685620,0.489142,91.729323,0.000000,91.729323,8.270677,7.620677,532,unseen_item_limited_history_transfer,14,19
7,1.127820,2.681214,47.694754,0.224492,0.563910,0.718158,0.502187,91.917293,0.000000,91.917293,8.082707,7.952820,532,unseen_item_limited_history_transfer,28,19
8,1.171992,2.910071,49.562798,0.227481,0.585996,0.633214,0.482231,92.293233,0.000000,92.293233,7.706767,7.174624,532,unseen_item_limited_history_transfer,56,19
9,2.393797,6.914998,101.232114,0.236466,1.196898,1.570602,1.001322,80.827068,0.000000,80.827068,19.172932,2.275564,532,unseen_item_zero_shot,0,19


## 3. Inspect the cohort sizes and separation

In [3]:
print("Training series:", len(split["training_series_ids"]))
print("Unseen-item series:", len(split["unseen_item_series_ids"]))
print("New-combination series:", len(split["combination_holdout_series_ids"]))
print("Unseen item IDs:", len(split["unseen_items"]))

Training series: 78
Unseen-item series: 19
New-combination series: 3
Unseen item IDs: 19


## 4. Limitations

Only three repeated items exist in the sampled 100 series, so the new-combination cohort is small. Zero-shot scale is estimated from training items in the same broad category; it does not use held-out target history. A larger M5 sample would give more reliable cold-start subgroup estimates.